# Experiments

This is where to decide whether an idea helps. Everything here runs on the **seven open matches**,
damaged to look like the real tracking, and each match is scored by a model that never saw it.
These numbers contain no data from the provided match, so they are safe to share.

**To try an idea:**

1. Change the code in `src/`.
2. If you changed the clues or the damage, run `python victor/train_model.py --cv` in the terminal. The
   old models would otherwise give a meaningless score, with no error.
3. Add a version in section 4 and rerun from there.

---

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import os, sys
sys.path[:0] = [os.path.abspath(p) for p in ('..', '../src', '../..', '../../starter', '../../evaluation')]

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import score_external as bed
import model, report, run, stage2

## 1. All versions

The run takes a minute or two. `worst_match` and `best_match` show how much a score moves between
matches.

In [ ]:
table = bed.score_versions()
bed.summary(table).round(4)

## 2. Is the gain real?

An average can hide one lucky match. A real improvement should win on most matches, so compare two
versions match by match.

In [ ]:
def compare(table, new, old):
    by_match = table.pivot_table(index='match', columns='version', values='spellF1')
    gain = (by_match[new] - by_match[old]).sort_values()
    print('%s against %s: mean %+.4f, better on %d of %d matches'
          % (new, old, gain.mean(), (gain > 0).sum(), len(gain)))
    fig, ax = plt.subplots(figsize=(8, 2.6))
    gain.plot.bar(ax=ax, color=['#a50044' if value > 0 else '#888888' for value in gain])
    ax.axhline(0, color='k', lw=.8); ax.set_ylabel('spellF1 gain')
    plt.show()

compare(table, 'current', 'rules')

## 3. Where are the points?

This shows what the current version would score with perfect dead-ball calls, and with perfect team
calls. The bigger gain is the better place to work.

In [ ]:
limits = bed.what_if(bed.current)
limits[['as_it_is', 'perfect_dead_ball', 'perfect_team']].mean().round(4)

## 4. Try an idea

A version is a function of `(data, blocks, match_id)` that returns labels. The example changes the
cost of a possession change. Replace it with your own idea.

In [ ]:
def higher_cost(data, blocks, match_id):
    return run.predict(data, blocks, model.load(model.cv_path(match_id)),
                       stage2.load(stage2.cv_path(match_id)), cost=20)

ideas = {'current': bed.current, 'higher_cost': higher_cost}
trial = bed.score_versions(ideas)
print(bed.summary(trial).round(4).to_string())
compare(trial, 'higher_cost', 'current')

## 5. Keep a log

Record what you tried, including what didn't work. The technical document has to cover assets you
dropped as well as those you kept. Uncomment the line below to add a row to
`experiments_log.csv`.

In [ ]:
# report.log_result('higher_cost', 'change cost 20 instead of 10', trial)

pd.read_csv(report.LOG_PATH) if os.path.isfile(report.LOG_PATH) else 'no log yet'